In [16]:
import pandas as pd

In [17]:
places_columns = [
    'geonameid', 'name', 'asciiname', 'alternatenames', 'latitude', 'longitude', 
    'feature_class', 'feature_code', 'country_code', 'cc2', 'admin1_code', 
    'admin2_code', 'admin3_code', 'admin4_code', 'population', 'elevation', 
    'dem', 'timezone', 'modification_date'
]

places_df = pd.read_csv('IN_Places/IN.txt', sep='\t', names=places_columns, low_memory=False)

print(f"Places data loaded! Total rows: {places_df.shape[0]}")

useful_cols = ['geonameid', 'name', 'asciiname', 'alternatenames', 
               'latitude', 'longitude', 'feature_class', 'feature_code',
               'admin1_code', 'admin2_code', 'population']

places_df=places_df[useful_cols]
print(places_df.head(3))

Places data loaded! Total rows: 659935
   geonameid           name      asciiname  \
0    1114940     Rāvi River     Ravi River   
1    1114942  Punjab Plains  Punjab Plains   
2    1114957   Jhelum River   Jhelum River   

                                      alternatenames  latitude  longitude  \
0                    Ravi,Ravi River,Rāvi,Rāvi River  30.62123   71.82683   
1                                      Punjab Plains  30.00000   75.00000   
2  Jhelum,Jhelum River,River Hydaspes,Veth River,...  31.16853   72.15066   

  feature_class feature_code  admin1_code admin2_code  population  
0             H          STM          0.0         NaN           0  
1             T          PLN          0.0         NaN           0  
2             H          STM          0.0         NaN           0  


In [18]:

states_df = places_df[
    (places_df['feature_class'] == 'A') & 
    (places_df['feature_code'] == 'ADM1')
].copy()

cities_df = places_df[
    places_df['feature_class'] == 'P'
].copy()

print(f"States: {len(states_df)}, Cities: {len(cities_df)}")
print("\nStates head")
print(states_df.head(3))
print("\nCities head")
print(cities_df.head(3))

States: 36, Cities: 557959

States head
      geonameid           name      asciiname  \
250     1252881    West Bengal    West Bengal   
993     1253626  Uttar Pradesh  Uttar Pradesh   
1533    1254169        Tripura        Tripura   

                                         alternatenames  latitude  longitude  \
250   Arevmtyan Bengalia,Bangla,Bati Bengal,Batis Be...     24.00      88.00   
993   Arewacin Pradesh,Bundesstaat Uttar Pradesh,Nor...     27.25      80.75   
1533  Bundesstaat Tripura,State of Tripura,TR,Thit-l...     24.00      92.00   

     feature_class feature_code  admin1_code admin2_code  population  
250              A         ADM1         28.0         NaN    91276115  
993              A         ADM1         36.0         NaN   199812341  
1533             A         ADM1         26.0         NaN     3673917  

Cities head
   geonameid     name asciiname alternatenames  latitude  longitude  \
5    1163293  Tithwāl   Tithwal            NaN  34.39351   73.77416   
6  

In [19]:
pin_columns = [
    'country_code', 'pincode', 'place_name',
    'state_name', 'state_code',
    'district_name', 'district_code',
    'community_name', 'community_code',
    'latitude', 'longitude', 'accuracy'
]

pin_df = pd.read_csv('IN_Pin/IN.txt', sep='\t', names=pin_columns, low_memory=False)

print(f"Pincodes loaded: {len(pin_df)}")
print("\nPincodes head")
print(pin_df.head(3))

Pincodes loaded: 155570

Pincodes head
  country_code  pincode place_name                 state_name  state_code  \
0           IN   744301      Sawai  Andaman & Nicobar Islands           1   
1           IN   744301     Kakana  Andaman & Nicobar Islands           1   
2           IN   744301        Mus  Andaman & Nicobar Islands           1   

  district_name  district_code community_name  community_code  latitude  \
0       Nicobar          638.0     Carnicobar             NaN    7.5166   
1       Nicobar          638.0     Carnicobar             NaN    9.1167   
2       Nicobar          638.0     Carnicobar             NaN    9.2333   

   longitude  accuracy  
0    93.6031         4  
1    92.8000         4  
2    92.7833         4  


In [20]:
admin1_lookup = dict(zip(states_df['admin1_code'], states_df['asciiname']))

print("Sample lookup:", list(admin1_lookup.items())[:5])

Sample lookup: [(28.0, 'West Bengal'), (36.0, 'Uttar Pradesh'), (26.0, 'Tripura'), (40.0, 'State of Telangana'), (25.0, 'State of Tamil Nadu')]


In [21]:
#we are making duplicate enteries for alt names
def expand_names(df, entity_type):
    result = []
    
    for i in range(len(df)):
        row = df.iloc[i]
        

        #to ignore repeated names set holds unique elements
        all_names = set()

        all_names.add(str(row['asciiname']).strip().lower())
        all_names.add(str(row['name']).strip().lower())
        
        # add alternate names
        if str(row['alternatenames']) != 'nan':
            alt_list = str(row['alternatenames']).split(',')
            for alt in alt_list:
                all_names.add(alt.strip().lower())
        
        all_names.discard('')
        
        #append all alt names in a single df
        for name in all_names:
            result.append({
                'search_term': name,
                'entity_type': entity_type,
                'entity_name': row['asciiname'],
                'latitude':    row['latitude'],
                'longitude':   row['longitude'],
                'geonameid':   row['geonameid'],
                'city':        row['name'] if entity_type == 'city' else None,
                'state':       row['name'] if entity_type == 'state' else row['state'],
                'pincode':     None,
                'population':  row['population'],
                'admin1_code': row['admin1_code']
            })
    
    return pd.DataFrame(result)

cities_df['state'] = cities_df['admin1_code'].map(admin1_lookup)

print("Building states...")
search_states = expand_names(states_df, 'state')

print("Building cities...")
search_cities = expand_names(cities_df, 'city')

print("States shape:", search_states.shape)
print("Cities shape:", search_cities.shape)


Building states...
Building cities...
States shape: (2258, 11)
Cities shape: (883913, 11)


In [22]:
#there are duplicate pin codes for different places so to take the major cities we pick the city same as district name

pin_main = pin_df[pin_df['place_name'] == pin_df['district_name']].drop_duplicates(subset='pincode', keep='first')

# some districts dont have cities with same name these cover those and just pick the first and frop all other with same pincode
pin_fallback = pin_df[~pin_df['pincode'].isin(pin_main['pincode'])].sort_values('accuracy').drop_duplicates(subset='pincode', keep='first')

pin_main = pd.concat([pin_main, pin_fallback], ignore_index=True)

In [27]:
pin_main['population'] = None

search_pins = pd.DataFrame({
    'search_term':  pin_main['pincode'].astype(str),
    'entity_type':  'pincode',
    'entity_name':  pin_main['pincode'].astype(str),
    'latitude':     pin_main['latitude'],
    'longitude':    pin_main['longitude'],
    'geonameid':    None,
    'city':         pin_main['district_name'],
    'state':        pin_main['state_name'],
    'pincode':      pin_main['pincode'].astype(str),
    'population':   None,
    'admin1_code':  None
})

print(search_pins[search_pins['pincode'] == '208001'])

      search_term entity_type entity_name  latitude  longitude geonameid  \
10471      208001     pincode      208001   26.6705    79.9757      None   

               city          state pincode population admin1_code  
10471  Kanpur Dehat  Uttar Pradesh  208001       None        None  


In [28]:
search_df = pd.concat([search_states, search_cities, search_pins], ignore_index=True)
print(f"Total search index size: {len(search_df)}")



Total search index size: 905409


C:\Users\tiwar\AppData\Local\Temp\ipykernel_11400\153296312.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  search_df = pd.concat([search_states, search_cities, search_pins], ignore_index=True)


In [31]:
print(search_df.head(3))

         search_term entity_type  entity_name  latitude  longitude geonameid  \
0        مغربی بنگال       state  West Bengal      24.0       88.0   1252881   
1        لیندا بنگال       state  West Bengal      24.0       88.0   1252881   
2  dasavleti bengali       state  West Bengal      24.0       88.0   1252881   

   city        state pincode population  admin1_code  
0  None  West Bengal    None   91276115         28.0  
1  None  West Bengal    None   91276115         28.0  
2  None  West Bengal    None   91276115         28.0  


In [30]:
import pickle

with open('search_df.pkl', 'wb') as f:
    pickle.dump(search_df, f)

print(f"Saved search_df with {len(search_df)} rows")

Saved search_df with 905409 rows
